# Experiment with local model code

Two independent examples: **feature selection** and **inference post-processing**.
Run the setup cell, then the example you need. Model functions run in this notebook's
Python process, using the packages installed in its kernel. No SageMaker job is started.
Use the same model dependencies as the approved runtime image.

The pattern is **prepare paths → set environment → import → call**. All outputs stay
under `.runs/model-experiments`. You can use another writable location, including
`/opt/ml`, by changing the example's root directory.

## Setup
- Locate this checkout and add its model source to Python's import path.
- Load standard notebook tools and the helper that downloads a job's inputs.
- Keep model imports for later, after their environment variables are set.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models/src"), str(ROOT / "mlops-models"),
               str(ROOT / "notebooks/source/03_experiment_model_base")]
from local_inputs import prepare_postprocess

print("Local checkout:", ROOT)
print("Notebook Python:", sys.executable)

## 1. Feature selection: prepare the three paths

`data_directory` contains `data.csv`. `output_directory` receives the selection
outputs. `parameter_path` points to the selected model's existing JSON file.

- Choose the model folder and create a fresh local input/output directory.
- Read `csv_input` from `notebooks/config.json`, accepting a local file or S3 object.
- Copy that CSV unchanged to `data_directory/data.csv`; keep the parameter file unchanged.
- Print all three paths so the inputs and output are visible before calling the model.

In [ ]:
import shutil
import boto3

MODEL_ID = "usecase1"  # Choose your actual model folder.
feature_root = ROOT / ".runs/model-experiments" / MODEL_ID / datetime.now(timezone.utc).strftime("selection-%Y%m%dT%H%M%S%f")
data_directory = feature_root / "input"
output_directory = feature_root / "output"
parameter_path = ROOT / "mlops-models/src/mlops_models/models" / MODEL_ID / "parameters.json"

config_path = ROOT / "notebooks/config.json"
csv_input = json.loads(config_path.read_text()).get("csv_input", "").strip()
if not csv_input:
    raise ValueError(f"Set csv_input in {config_path} to your local CSV or S3 object.")
if not parameter_path.is_file():
    raise FileNotFoundError(parameter_path)
data_directory.mkdir(parents=True)
output_directory.mkdir()
local_csv = data_directory / "data.csv"
if csv_input.startswith("s3://"):
    bucket, key = csv_input.removeprefix("s3://").split("/", 1)
    with boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"] as source, local_csv.open("wb") as destination:
        shutil.copyfileobj(source, destination)
else:
    source = Path(csv_input).expanduser()
    if not source.is_absolute():
        source = ROOT / source
    shutil.copy2(source, local_csv)

print("data_directory:", data_directory)
print("output_directory:", output_directory)
print("parameter_path:", parameter_path)

### Set the environment and read the inputs
- Set the model and output paths before importing feature-selection code.
- Disable cloud tracking and run-report writes for this local experiment.
- Load the existing JSON parameters and read `data_directory/data.csv` using the model's reader.
- Display a few rows; the selector below receives the full dataframe.

In [ ]:
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_MODEL_DIR": str(parameter_path.parent),
    "SM_MODEL_DIR": str(parameter_path.parent),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_TRAINING_MODE": "1",
    "MLOPS_MODEL_METADATA_URI": "none",
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})
parameters = json.loads(parameter_path.read_text())
from mlops_models.model_base.training import feature_selection
feature_selection = importlib.reload(feature_selection)
print("Feature-selection code:", feature_selection.__file__)
df = feature_selection.get_container_input(data_directory, parameters)
display(df.head())

### Fit and inspect the selector
- Construct the selector with the parameters loaded from `parameter_path`.
- Fit it on the dataframe returned by `get_container_input`.
- Export the selected features to `output_directory/selection` and display the result.

These are the same steps performed by `run_feature_selection(data_directory,
output_directory, parameters, selector_cls=feature_selection.FeatureSelector)`.
They are shown separately so you can inspect or change the dataframe and selector.
After editing the module, rerun the previous import cell and this cell.

In [ ]:
import pandas as pd

selector = feature_selection.FeatureSelector(parameters)
selector.fit(df)
selection_directory = feature_selection.write_container_output(selector, output_directory)
display(pd.read_csv(selection_directory / "selected_features.csv"))
print("Selection output:", selection_directory)

## 2. Inference post-processing: download matching inputs

Run this section after **Setup**; it does not need the feature-selection section.
Supply one inference execution ARN. An execution ID also works if notebook 02's
publication is saved for the configured namespace.

The helper reads that execution's **Postprocess Processing job inputs**, including
its actual model archive. It does not look up the latest approved model. A failed
Postprocess job works too, provided the job was created and its inputs still exist.
The helper uses read-only SageMaker/S3 calls; model code is imported from this checkout.

- Select the inference execution whose post-processing you want to reproduce.
- Download its raw data, selected features, predictions and model archive into a fresh directory.
- Unpack the model and create a writable output directory, preserving nested prediction filenames.
- Save `paths.json` so these same local inputs can be reused from VS Code without another download.

In [ ]:
INFERENCE_EXECUTION = ""  # Paste the full inference execution ARN, or its execution ID.
postprocess_root = ROOT / ".runs/model-experiments" / datetime.now(timezone.utc).strftime("postprocess-%Y%m%dT%H%M%S%f")

config_path = ROOT / "notebooks/config.json"
config = json.loads(config_path.read_text()) if config_path.is_file() else {}
publication_path = ROOT / ".runs/source-deploy" / config.get("namespace", "") / "publication.json"
publication = json.loads(publication_path.read_text()) if publication_path.is_file() else {}
pipeline_arn = publication.get("pipelines", {}).get("inference", {}).get("arn", "")
inputs = prepare_postprocess(INFERENCE_EXECUTION, postprocess_root, pipeline_arn=pipeline_arn)
print("Execution:", inputs["execution_arn"])
print("Postprocess job:", inputs["job_arn"])
print("VS Code INPUTS_FILE:", postprocess_root / "paths.json")

### Expose paths and set the environment
- Assign the five paths and analysis mode passed to `postprocess_predictions`.
- Set the model-directory environment variables to the unpacked model's parameters.
- Set the output environment variables to the local output directory before importing the module.
- Disable tracking/report uploads; local post-processing will write to the printed output directory.

In [ ]:
data_directory = Path(inputs["data_directory"])           # Contains data.csv.
selection_file = Path(inputs["selection_file"])           # A CSV file, not a directory.
prediction_directory = Path(inputs["prediction_directory"])  # Contains prediction CSVs.
output_directory = Path(inputs["output_directory"])
model_directory = Path(inputs["model_directory"])         # Unpacked model and parameters.
analysis_mode = inputs["analysis_mode"]

os.environ.update({
    "MODEL_ID": inputs["model_id"],
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_ANALYSIS_MODE": analysis_mode,
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})
for name in ("data_directory", "selection_file", "prediction_directory", "output_directory", "model_directory", "analysis_mode"):
    print(name, "=", inputs[name])

### Run or debug the local post-processing code
- Import the local base module after setting its environment variables.
- Reload it so rerunning this cell picks up edits to that module.
- Call `postprocess_predictions` directly with the prepared paths and analysis mode.

Set breakpoints inside that function and use the notebook cell's debugger. Alternatively,
open `notebooks/source/03_experiment_model_base/debug_postprocess.py`, set `INPUTS_FILE`
to the printed `paths.json`, then **Debug Python File**. That script uses the same local
inputs and makes the same function call. Use a Python environment with the model's dependencies.
Restart the kernel after switching models or editing dependencies loaded by another module.

In [ ]:
from mlops_models.model_base.inference import postprocess
postprocess = importlib.reload(postprocess)
print("Post-processing code:", postprocess.__file__)
postprocess.postprocess_predictions(
    data_directory=data_directory,
    selection_file=selection_file,
    prediction_directory=prediction_directory,
    output_directory=output_directory,
    model_directory=model_directory,
    analysis_mode=analysis_mode,
)

### Inspect the outputs
- List the files written by your post-processing code.
- Display the example analysis CSV when that file exists.
- Keep the prepared inputs for another local edit/run cycle; no new SageMaker execution is needed.

In [ ]:
for output_file in sorted(output_directory.rglob("*")):
    if output_file.is_file():
        print(output_file)
analysis_csv = output_directory / "prediction_analysis.csv"
if analysis_csv.is_file():
    import pandas as pd
    saved_parameters = json.loads((model_directory / "parameters.json").read_text())
    display(pd.read_csv(analysis_csv, dtype={saved_parameters["primary_key"]: str}).head())